# 实验任务一: Transformer

## **1. Transformer 编码器中 Encoder Layer 的实现**
Encoder Layer简介
    Encoder Layer 是 Transformer 编码器中的基本构建单元，由多头自注意力机制（Multi-Head Self-Attention）和前馈全连接网络（Feed Forward Network）组成，搭配两次残差连接与 LayerNorm，用于高效建模输入序列的上下文依赖关系和特征表达能力。

这次我们仍然使用 AG News 数据集完成分类任务。

实验要求：
1. 显式构造并传入 attention_mask。
2. 比较 mean pooling 与 max pooling。
3. 展示一层注意力权重的 shape 或可视化结果。

选做内容：
1. 固定 num_layers=2，比对 n_heads=2 与 n_heads=4。

首先导入所需模块：


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
# from torchtext.data.utils import get_tokenizer
# from torchtext.vocab import build_vocab_from_iterator
from tqdm import tqdm
import pandas as pd
from transformers import AutoTokenizer

数据读取以及预处理

不同于上次，这次我们使用pandas读取数据，相应的代码也有所修改。

In [ ]:
# **1. 加载 AG NEWS 数据集**
df = pd.read_csv("train.csv")  # 请替换成你的文件路径
df.columns = ["label", "title", "description"]
df["text"] = df["title"] + " " + df["description"]
df["label"] = df["label"] - 1
train_texts, train_labels = df["text"].tolist(), df["label"].tolist()
number = int(0.3 * len(train_texts))
train_texts, train_labels = train_texts[: number], train_labels[: number]

df = pd.read_csv("test.csv")  # 请替换成你的文件路径
df.columns = ["label", "title", "description"]
df["text"] = df["title"] + " " + df["description"]
df["label"] = df["label"] - 1
test_texts, test_labels = df["text"].tolist(), df["label"].tolist()

# **2. 加载 BERT Tokenizer**
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# **3. 处理数据**
class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

vocab = tokenizer.get_vocab()
pad_idx = tokenizer.pad_token_id
unk_idx = tokenizer.unk_token_id

train_dataset = AGNewsDataset(train_texts, train_labels, tokenizer)
test_dataset = AGNewsDataset(test_texts, test_labels, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)


### 位置编码器
在 Transformer 模型中，Self-Attention 是无序的，它无法感知输入序列的「位置信息」，即每个 token 在序列中的先后顺序。

相比之下，RNN（通过时间步）和 CNN（通过局部感受野）天然就有顺序/位置的概念。

因此，Transformer 需要为每个 token embedding 加入「位置信息」，这就是 Positional Encoding 的作用。

论文《Attention is All You Need》中提出了如下的位置编码方式，使用正弦和余弦函数来构造具有不同频率的位置表示。

原始公式：

$$
PE(pos, 2i) = \sin\left( \frac{pos}{10000^{\frac{2i}{d_{model}}}} \right)
$$

$$
PE(pos, 2i+1) = \cos\left( \frac{pos}{10000^{\frac{2i}{d_{model}}}} \right)
$$

其中：

-  $pos$：是序列中的位置$（0, 1, 2, ..., L-1）$
-  $i$：是 embedding 的维度索引$（0 ~ d_{model}/2）$
-  $d_{model}$：是 embedding 的总维度
-  $10000$：是一个控制不同维度的 $sin/cos$ 波动频率的超参数


公式进一步推导，
原公式中：

$$
\frac{pos}{10000^{\frac{2i}{d_{model}}}}
$$

等价于：

$$
pos \times \frac{1}{10000^{\frac{2i}{d_{model}}}}
$$

进一步展开成指数形式：

$$
= pos \times e^{- \log(10000) \cdot \frac{2i}{d_{model}}}
$$

请参考展开后的指数形式补充完下面的代码：

In [ ]:
#这段代码是 Transformer中的位置编码（PositionalEncoding），用于给输入的 token embedding 加入位置信息。
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # 创建一个全0的矩阵，shape = (max_len, d_model)
        # 表示每个位置 (0 ~ max_len-1) 对应的 d_model 维位置编码
        pe = torch.zeros(max_len, d_model)

        # 生成位置索引，shape = (max_len, 1)
        # 即 position = [0, 1, 2, ..., max_len-1] 的列向量
        position = torch.arange(0, max_len).unsqueeze(1)

        # TODO 1: 计算 div_term，用于控制不同维度的 sin/cos 频率
        # 要求: 使用 torch.exp() 实现 1 / 10000^(2i/d_model)
        div_term = ...

        # TODO 2: 给偶数维度位置编码赋值
        # 要求: 使用 torch.sin() 完成 position * div_term，赋值给 pe 的偶数列
        pe[:, 0::2] = ...

        # TODO 3: 给奇数维度位置编码赋值
        # 要求: 使用 torch.cos() 完成 position * div_term，赋值给 pe 的奇数列
        pe[:, 1::2] = ...

        # 将 pe 注册为 buffer（不会被训练优化）
        # 并扩展成 (1, max_len, d_model) 方便后续和 batch 做广播
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: (1, max_len, d_model)

    def forward(self, x):
        # x 是输入的 embedding，shape = (batch_size, seq_len, d_model)

        # 将对应位置的 pe 加到 x 上
        # self.pe[:, :x.size(1)] shape = (1, seq_len, d_model) 自动广播到 batch_size
        x = x + self.pe[:, :x.size(1)]

        # 返回位置编码后的 embedding
        return x

思考题1：为什么 padding mask 不能在加完位置编码后，再通过“某个位置的向量是否全零”来判断？

### Multi-Head Self-Attention 模块
下面是多头自注意力的实现，请你按照要求补全代码：


In [ ]:
# Multi-Head Self-Attention 的完整实现
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        # 这里共用一个线性层，一次性得到 Q、K、V
        self.qkv_linear = nn.Linear(d_model, d_model * 3)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None, return_attn=False):
        # x: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.size()

        # 线性映射后再切成多头
        qkv = self.qkv_linear(x)
        qkv = qkv.view(batch_size, seq_len, self.n_heads, 3 * self.d_k)
        qkv = qkv.permute(0, 2, 1, 3)

        # q, k, v: (batch_size, n_heads, seq_len, d_k)
        q, k, v = qkv.chunk(3, dim=-1)

        # TODO 1: 使用缩放点积计算 attention scores
        # 提示：K 需要先转置最后两个维度，再除以 sqrt(d_k)
        scores = ...

        # mask 中 1 表示有效 token，0 表示 padding
        # 被 mask 掉的位置应替换成一个很小的数，这样 softmax 后权重接近 0
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # TODO 2: 在 seq_len 维度上计算 attention 权重
        attn = ...

        # TODO 3: 用 attention 权重对 V 做加权求和
        context = ...

        # 拼回多头结果
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.fc(context)

        # 额外要求：支持返回注意力权重，方便结果展示
        if return_attn:
            return output, attn
        return output


思考题2：为什么我们需要使用多个 attention head？

思考题3：为什么要用缩放因子 sqrt(d_k)？

### TransformerEncoderLayer
下面的代码实现了 Transformer 编码器中的一个标准 Encoder Layer，请你按照要求补全代码：


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None, return_attn=False):
        # TODO 1: 先经过自注意力层
        # 提示：当 return_attn=True 时，需要同时接收 attn，方便后续展示注意力结果
        if return_attn:
            x2, attn = ...
        else:
            x2 = ...
            attn = None

        # TODO 2: 残差连接 + 第一层 LayerNorm
        # 提示：这里通常是原输入 x 与自注意力输出 x2 相加后再做归一化
        x = ...

        # TODO 3: 前馈网络
        x2 = ...

        # TODO 4: 残差连接 + 第二层 LayerNorm
        x = ...

        if return_attn:
            return x, attn
        return x


思考题4：为什么 Transformer Encoder Layer 中要在 Self-Attention 和 Feed Forward Network 之后都使用残差连接和 LayerNorm？请从训练稳定性和特征传递两个角度进行分析。

## **2. 基于 Transformer Encoder 的文本分类器**
下面，我们实现一个基于 Transformer Encoder 的文本分类器。
请注意：
1. 需要显式传入 attention_mask。
2. 必做是比较 mean pooling 和 max pooling。
3. attention head 数量对比属于选做，不要求所有同学都完成。


In [ ]:
class TransformerEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=256, num_layers=2, num_classes=4, pooling="mean"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([TransformerEncoderLayer(d_model, n_heads, d_ff) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, num_classes)
        self.pooling = pooling

    def forward(self, input_ids, attention_mask=None, return_attn=False):
        # input_ids: (batch_size, seq_len)
        x = self.embedding(input_ids)
        x = self.pos_encoder(x)

        # 这里要求显式传入 attention_mask
        # 如果没有传入，这里给出一个基于 pad_idx 的兜底写法
        if attention_mask is None:
            attention_mask = (input_ids != pad_idx).long()

        # 给 self-attention 用的 mask 形状通常是 (batch, 1, 1, seq_len)
        pad_mask = attention_mask.unsqueeze(1).unsqueeze(2)

        # 如果需要展示注意力结果，可以取最后一层的注意力权重
        last_attn = None
        for idx, layer in enumerate(self.layers):
            if return_attn and idx == len(self.layers) - 1:
                x, last_attn = layer(x, pad_mask, return_attn=True)
            else:
                x = layer(x, pad_mask)

        # pooling 时也要考虑 padding 位置
        token_mask = attention_mask.unsqueeze(-1)
        if self.pooling == "mean":
            # TODO 1: 只对有效 token 做平均池化
            out = ...
        elif self.pooling == "max":
            # TODO 2: 先屏蔽 padding，再做最大池化
            out = ...
        else:
            raise ValueError(f"Unsupported pooling: {self.pooling}")

        logits = self.fc(out)
        if return_attn:
            return logits, last_attn
        return logits


思考题5：mean pooling 和 max pooling 分别更容易保留哪类信息？在文本分类任务中它们各自可能有什么优缺点？

下面是模型的训练和测试：


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()


模型训练部分：

In [ ]:
def train_epoch(model, optimizer):
    model.train()
    total_loss = 0
    loop = tqdm(train_dataloader, desc="Training", leave=False)

    for batch in loop:
        # 这里的 batch 是一个字典，包含 input_ids、attention_mask、labels
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        output = model(input_ids, attention_mask=attention_mask)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_dataloader)


模型测试部分：

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    loop = tqdm(test_dataloader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            output = model(input_ids, attention_mask=attention_mask)
            preds = output.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


# 必做：比较 mean pooling 与 max pooling
experiment_settings = [
    {"pooling": "mean", "n_heads": 4},
    {"pooling": "max", "n_heads": 4},
]

results = []
for config in experiment_settings:
    model = TransformerEncoderClassifier(
        len(vocab),
        n_heads=config["n_heads"],
        num_layers=2,
        pooling=config["pooling"],
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, 4):
        loss = train_epoch(model, optimizer)
        acc = evaluate(model)
        print(f'{config}, Epoch {epoch}: Loss = {loss:.4f}, Test Acc = {acc:.4f}')

    results.append({"pooling": config["pooling"], "n_heads": config["n_heads"], "acc": acc})

print(results)

# 选做：固定 pooling 后，再比较 n_heads=2 与 n_heads=4 的差异
# head_experiment_settings = [
#     {"pooling": "mean", "n_heads": 2},
#     {"pooling": "mean", "n_heads": 4},
# ]


### 注意力权重展示（必做）
在完成训练后，请运行下面的单元，展示最后一层注意力权重的 shape，或在此基础上进一步可视化。


In [ ]:
sample_batch = next(iter(test_dataloader))
sample_input_ids = sample_batch["input_ids"][:1].to(device)
sample_attention_mask = sample_batch["attention_mask"][:1].to(device)

logits, attn = model(
    sample_input_ids,
    attention_mask=sample_attention_mask,
    return_attn=True,
)
print(attn.shape)


思考题6：为什么在本实验中，更复杂的设置不一定总能带来更好的结果？请结合 pooling 或 attention head 的实验现象简要分析。
